# 01 Data Gathering

Notebook ini memecah alur gathering dari notebook eksplorasi lama menjadi pipeline terpisah yang lebih rapi. Fokus tahap ini hanya membaca data raw, menyatukan struktur minimum, memvalidasi bentuk awal, dan menyimpan dataset gabungan.

## Setup Environment

Tujuan: menyiapkan library, root project, path data, dan konfigurasi display.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

try:
    from fingo_ds1.config import RAW_DATA_PATH, INTERIM_DATA_PATH
except ImportError:
    RAW_DATA_PATH = project_root / 'data' / 'raw'
    INTERIM_DATA_PATH = project_root / 'data' / 'interim'

raw_data_path = Path(RAW_DATA_PATH)
interim_data_path = Path(INTERIM_DATA_PATH)
interim_data_path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(f'Project root: {project_root}')
print(f'Raw data path: {raw_data_path}')
print(f'Interim data path: {interim_data_path}')

Project root: /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Raw data path: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw
Interim data path: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim


## Reusable Data Loading Helpers

Tujuan: membuat loader bersih untuk CSV dan Excel agar semua sumber raw dapat dibaca dengan pola yang sama.

In [2]:
supported_extensions = {'.csv', '.xlsx', '.xls'}
excel_namespace = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}


def excel_column_index(cell_reference):
    letters = ''.join(character for character in str(cell_reference) if character.isalpha()) or 'A'
    index = 0
    for letter in letters.upper():
        index = index * 26 + ord(letter) - ord('A') + 1
    return index - 1


def read_excel_without_engine(file_path):
    with ZipFile(file_path) as archive:
        names = set(archive.namelist())
        shared_strings = []

        if 'xl/sharedStrings.xml' in names:
            root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            shared_strings = [
                ''.join(text.text or '' for text in item.findall('.//main:t', excel_namespace))
                for item in root.findall('main:si', excel_namespace)
            ]

        sheet = ET.fromstring(archive.read('xl/worksheets/sheet1.xml'))
        rows = []
        width = 0

        for row in sheet.findall('.//main:sheetData/main:row', excel_namespace):
            values = []
            for cell in row.findall('main:c', excel_namespace):
                column_index = excel_column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column_index:
                    values.append(np.nan)

                cell_type = cell.attrib.get('t')
                if cell_type == 'inlineStr':
                    value = ''.join(text.text or '' for text in cell.findall('.//main:t', excel_namespace))
                else:
                    raw_value = cell.find('main:v', excel_namespace)
                    value = raw_value.text if raw_value is not None else np.nan
                    if cell_type == 's' and pd.notna(value):
                        value = shared_strings[int(value)]

                values[column_index] = value

            width = max(width, len(values))
            rows.append(values)

        if not rows:
            return pd.DataFrame()

        rows = [row + [np.nan] * (width - len(row)) for row in rows]
        columns = [f'column_{index}' if pd.isna(value) or value == '' else value for index, value in enumerate(rows[0])]
        return pd.DataFrame(rows[1:], columns=columns)


def read_data_file(file_path):
    file_path = Path(file_path)

    if file_path.name.startswith('~$'):
        return None

    try:
        if file_path.suffix.lower() == '.csv':
            return pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')

        if file_path.suffix.lower() in {'.xlsx', '.xls'}:
            try:
                return pd.read_excel(file_path)
            except ImportError:
                return read_excel_without_engine(file_path)

        return pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')
    except Exception as error:
        print(f'Error loading {file_path}: {error}')
        return None


def get_data_files(candidate):
    candidate = Path(candidate)

    if candidate.is_file():
        return [candidate]

    if candidate.is_dir():
        return sorted(
            path
            for path in candidate.rglob('*')
            if path.is_file() and path.suffix.lower() in supported_extensions and not path.name.startswith('~$')
        )

    print(f'Missing path skipped: {candidate}')
    return []


def load_dataset_group(candidates, source_name):
    frames = []
    loaded_files = 0

    for candidate in candidates:
        for file_path in get_data_files(candidate):
            frame = read_data_file(file_path)
            if frame is None or frame.empty:
                continue

            frame = frame.copy()
            frame['_source_file'] = file_path.name
            frame['_source_path'] = str(file_path.relative_to(project_root)) if file_path.is_relative_to(project_root) else str(file_path)
            frames.append(frame)
            loaded_files += 1

    if not frames:
        print(f'No data loaded for {source_name}')
        return pd.DataFrame()

    print(f'{source_name}: {loaded_files} files loaded')
    return pd.concat(frames, ignore_index=True, sort=False)


def load_first_available(candidates, source_name):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            frame = read_data_file(candidate)
            if frame is not None:
                print(f'{source_name} loaded from: {candidate}')
                return frame

    print(f'No data loaded for {source_name}')
    return pd.DataFrame()

## Load Household Transaction Data

Tujuan: membaca data household dari folder raw, termasuk file transaksi harian dan file Excel bulanan yang tersedia.

In [3]:
household_path = raw_data_path / 'daily_household_transaction'
household_candidates = [
    household_path / 'daily_household_transaction_clean',
    household_path / 'daily_household_transaction_raw_public',
    household_path / 'Daily_household_transactioncsv',
    household_path / 'Daily Household Transactions.csv',
]

df_household = load_dataset_group(household_candidates, 'household_transaction')
print(f'Household data: {len(df_household):,} records')
display(df_household.head())

Missing path skipped: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/daily_household_transaction/Daily_household_transactioncsv
household_transaction: 49 files loaded
Household data: 49,567 records


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_source_file,_source_path,product_category,Jumlah,Returned quantity,Total Berat,Waktu Pengiriman Diatur,Status Pembatalan/ Pengembalian,No. Resi,Antar ke counter/ pick-up,Pesanan Harus Dikirimkan Sebelum (Menghindari keterlambatan),Waktu Pembayaran Dilakukan,SKU Induk,Nomor Referensi SKU,Nama Variasi,Harga Awal,Harga Setelah Diskon,Total Harga Produk,Diskon Dari Penjual,Diskon Dari Shopee,Berat Produk,Jumlah Produk di Pesan,Voucher Ditanggung Penjual,Cashback Koin,Voucher Ditanggung Shopee,Paket Diskon,Paket Diskon (Diskon dari Shopee),Paket Diskon (Diskon dari Penjual),Potongan Koin Shopee,Diskon Kartu Kredit,Ongkos Kirim Pengembalian Barang,Catatan dari Pembeli,Catatan,Username (Pembeli),Nama Penerima,No. Telepon,Alamat Pengiriman,Waktu Pesanan Selesai,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Load Ecommerce Sales Data

Tujuan: membaca data ecommerce clean dan raw public dari struktur folder raw.

In [4]:
ecommerce_path = raw_data_path / 'Indonesian_Ecommerce_sales'
ecommerce_candidates = [
    ecommerce_path / 'Indonesian_Ecommerce_sales_clean',
    ecommerce_path / 'Indonesian_Ecommerce_sales_raw_public',
]

df_ecommerce = load_dataset_group(ecommerce_candidates, 'ecommerce_sales')
print(f'Ecommerce data: {len(df_ecommerce):,} records')
display(df_ecommerce.head())

ecommerce_sales: 48 files loaded
Ecommerce data: 47,106 records


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_source_file,_source_path,product_category,Jumlah,Returned quantity,Total Berat,Waktu Pengiriman Diatur,Status Pembatalan/ Pengembalian,No. Resi,Antar ke counter/ pick-up,Pesanan Harus Dikirimkan Sebelum (Menghindari keterlambatan),Waktu Pembayaran Dilakukan,SKU Induk,Nomor Referensi SKU,Nama Variasi,Harga Awal,Harga Setelah Diskon,Total Harga Produk,Diskon Dari Penjual,Diskon Dari Shopee,Berat Produk,Jumlah Produk di Pesan,Voucher Ditanggung Penjual,Cashback Koin,Voucher Ditanggung Shopee,Paket Diskon,Paket Diskon (Diskon dari Shopee),Paket Diskon (Diskon dari Penjual),Potongan Koin Shopee,Diskon Kartu Kredit,Ongkos Kirim Pengembalian Barang,Catatan dari Pembeli,Catatan,Username (Pembeli),Nama Penerima,No. Telepon,Alamat Pengiriman,Waktu Pesanan Selesai
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Load Monthly and Personal Finance Data

Tujuan: membaca dataset bulanan dan personal finance sebagai sumber tambahan untuk gabungan raw.

In [5]:
df_monthly = load_first_available(
    [
        raw_data_path / 'all_months_clean.csv',
        raw_data_path / 'Indonesian_Ecommerce_sales' / 'all_months_clean.csv',
        project_root / 'data' / 'all_months' / 'all_months_clean.csv',
    ],
    'monthly_clean',
)

print(f'Monthly data: {len(df_monthly):,} records')

df_personal = load_first_available(
    [raw_data_path / 'personal_finance' / 'Personal_Finance_Dataset.csv'],
    'personal_finance',
)

print(f'Personal finance data: {len(df_personal):,} records')
display(df_monthly.head())

monthly_clean loaded from: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/Indonesian_Ecommerce_sales/all_months_clean.csv
Monthly data: 20,848 records
personal_finance loaded from: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/personal_finance/Personal_Finance_Dataset.csv
Personal finance data: 1,500 records


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024.xlsx
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024.xlsx
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024.xlsx
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024.xlsx
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024.xlsx


## Inspect Source Structures

Tujuan: melihat shape dan kolom setiap sumber sebelum standardisasi.

In [6]:
datasets = {
    'household': df_household,
    'ecommerce': df_ecommerce,
    'monthly': df_monthly,
    'personal': df_personal,
}

for name, frame in datasets.items():
    print(f'\n{name.upper()} columns:')
    print(frame.columns.tolist())
    print(f'Shape: {frame.shape}')


HOUSEHOLD columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', '_source_file', '_source_path', 'product_category', 'Jumlah', 'Returned quantity', 'Total Berat', 'Waktu Pengiriman Diatur', 'Status Pembatalan/ Pengembalian', 'No. Resi', 'Antar ke counter/ pick-up', 'Pesanan Harus Dikirimkan Sebelum (Menghindari keterlambatan)', 'Waktu Pembayaran Dilakukan', 'SKU Induk', 'Nomor Referensi SKU', 'Nama Variasi', 'Harga Awal', 'Harga Setelah Diskon', 'Total Harga Produk', 'Diskon Dari Penjual', 'Diskon Dari Shopee', 'Berat Produk', 'Jumlah Produk di Pesan', 'Voucher Ditanggung Penjual', 'Cashback Koin', 'Voucher Ditanggung Shopee', 'Paket Diskon', 'Paket Di

## Standardize Column Names

Tujuan: menyeragamkan nama kolom dan menandai asal data setiap record.

In [7]:
def standardize_columns(frame, source_name):
    frame = frame.copy()

    if frame.empty:
        frame['data_source'] = source_name
        return frame

    frame.columns = (
        pd.Index(frame.columns)
        .astype(str)
        .str.replace(r'^\ufeff', '', regex=True)
        .str.lower()
        .str.strip()
        .str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
        .str.strip('_')
    )
    frame['data_source'] = source_name
    return frame


df_household = standardize_columns(df_household, 'household_transaction')
df_ecommerce = standardize_columns(df_ecommerce, 'ecommerce_sales')
df_monthly = standardize_columns(df_monthly, 'monthly_clean')
df_personal = standardize_columns(df_personal, 'personal_finance')

print('Columns standardized')

Columns standardized


## Identify Common Schema

Tujuan: menentukan kolom inti yang wajib ada pada dataset gabungan.

In [8]:
required_columns = ['date', 'amount', 'category', 'description']
unique_columns = sorted(set().union(*(set(frame.columns) for frame in [df_household, df_ecommerce, df_monthly, df_personal])))

print(f'Total unique columns: {len(unique_columns)}')
print(f'\nAll columns:\n{unique_columns}')
print(f'\nRequired columns for alignment: {required_columns}')

Total unique columns: 67

All columns:
['alamat_pengiriman', 'alasan_pembatalan', 'amount', 'antar_ke_counter_pick_up', 'berat_produk', 'cashback_koin', 'catatan', 'catatan_dari_pembeli', 'category', 'currency', 'data_source', 'date', 'diskon_dari_penjual', 'diskon_dari_shopee', 'diskon_kartu_kredit', 'estimasi_potongan_biaya_pengiriman', 'harga_awal', 'harga_setelah_diskon', 'income_expense', 'jumlah', 'jumlah_produk_di_pesan', 'kota_kabupaten', 'metode_pembayaran', 'mode', 'nama_penerima', 'nama_variasi', 'no_resi', 'no_telepon', 'nomor_referensi_sku', 'note', 'num_product_categories', 'ongkos_kirim_dibayar_oleh_pembeli', 'ongkos_kirim_pengembalian_barang', 'opsi_pengiriman', 'order_id', 'paket_diskon', 'paket_diskon_diskon_dari_penjual', 'paket_diskon_diskon_dari_shopee', 'perkiraan_ongkos_kirim', 'pesanan_harus_dikirimkan_sebelum_menghindari_keterlambatan', 'potongan_koin_shopee', 'product_categories', 'product_category', 'provinsi', 'returned_quantity', 'sku_induk', 'source_file',

## Map Source Columns

Tujuan: memetakan variasi nama kolom dari setiap sumber ke schema inti.

In [9]:
column_mappings = {
    'household_transaction': {
        'transaction_date': 'date',
        'total_amount': 'amount',
        'product_category': 'category',
        'product_categories': 'category',
        'product_name': 'description',
        'note': 'description',
        'waktu_pesanan_dibuat': 'date',
        'total_pembayaran': 'amount',
        'order_id': 'description',
    },
    'ecommerce_sales': {
        'order_date': 'date',
        'price': 'amount',
        'product_category': 'category',
        'product_categories': 'category',
        'product_name': 'description',
        'waktu_pesanan_dibuat': 'date',
        'total_pembayaran': 'amount',
        'order_id': 'description',
    },
    'monthly_clean': {
        'waktu_pesanan_dibuat': 'date',
        'total_pembayaran': 'amount',
        'product_categories': 'category',
        'order_id': 'description',
    },
    'personal_finance': {
        'transaction_description': 'description',
    },
}


def map_columns(frame, source_name):
    frame = frame.copy()

    for source_column, target_column in column_mappings.get(source_name, {}).items():
        if source_column not in frame.columns or source_column == target_column:
            continue

        if target_column not in frame.columns:
            frame = frame.rename(columns={source_column: target_column})
        else:
            frame[target_column] = frame[target_column].where(frame[target_column].notna(), frame[source_column])

    return frame.loc[:, ~frame.columns.duplicated()]


df_household = map_columns(df_household, 'household_transaction')
df_ecommerce = map_columns(df_ecommerce, 'ecommerce_sales')
df_monthly = map_columns(df_monthly, 'monthly_clean')
df_personal = map_columns(df_personal, 'personal_finance')

print('Columns mapped')

Columns mapped


## Create Unified Schema

Tujuan: memastikan semua dataframe punya kolom inti dan tetap membawa kolom tambahan untuk tahap berikutnya.

In [10]:
def align_schema(frame, required_columns):
    frame = frame.copy()

    for column in required_columns:
        if column not in frame.columns:
            frame[column] = np.nan

    if 'data_source' not in frame.columns:
        frame['data_source'] = 'unknown'

    extra_columns = [column for column in frame.columns if column not in required_columns and column != 'data_source']
    return frame[required_columns + ['data_source'] + extra_columns]


aligned_frames = [
    align_schema(df_household, required_columns),
    align_schema(df_ecommerce, required_columns),
    align_schema(df_monthly, required_columns),
    align_schema(df_personal, required_columns),
]

for name, frame in zip(['Household', 'Ecommerce', 'Monthly', 'Personal'], aligned_frames):
    print(f'{name}: {frame.shape}')

Household: (49567, 65)
Ecommerce: (47106, 57)
Monthly: (20848, 20)
Personal: (1500, 6)


## Combine All Sources

Tujuan: menggabungkan semua sumber yang sudah aligned menjadi satu dataframe raw gabungan.

In [11]:
df_combined = pd.concat([frame for frame in aligned_frames if not frame.empty], ignore_index=True, sort=False)

print(f'Combined dataset shape: {df_combined.shape}')
print('\nRecords per source:')
print(df_combined['data_source'].value_counts())

Combined dataset shape: (119021, 66)

Records per source:
data_source
household_transaction    49567
ecommerce_sales          47106
monthly_clean            20848
personal_finance          1500
Name: count, dtype: int64


## Validate Combined Dataset

Tujuan: mengecek missing values, duplikasi, tipe data, tanggal, dan amount sebelum file disimpan.

In [12]:
missing = df_combined.isnull().sum()
missing_summary = (
    pd.DataFrame(
        {
            'missing_count': missing,
            'missing_pct': (missing / len(df_combined) * 100).round(2),
        }
    )
    .query('missing_count > 0')
    .sort_values('missing_pct', ascending=False)
)

print('=== DATA VALIDATION ===\n')
print(f'Total records: {len(df_combined):,}')
print(f'Total columns: {len(df_combined.columns)}')
print(f'Duplicate rows: {df_combined.duplicated().sum():,}')
print('\nMissing values:')
print(missing_summary)
print('\nData types:')
print(df_combined.dtypes)

=== DATA VALIDATION ===

Total records: 119,021
Total columns: 66


Duplicate rows: 4,933

Missing values:
                                                    missing_count  missing_pct
type                                                       117521        98.74
subcategory                                                117195        98.47
mode                                                       116560        97.93
income_expense                                             116560        97.93
currency                                                   116560        97.93
no_resi                                                    116081        97.53
antar_ke_counter_pick_up                                   116081        97.53
potongan_koin_shopee                                       116081        97.53
diskon_kartu_kredit                                        116081        97.53
paket_diskon_diskon_dari_penjual                           116081        97.53
status_pembatalan_pengembalian                             116081        97.53
pesanan_harus

## Validate Date and Amount

Tujuan: menstandarkan tipe tanggal dan amount untuk memastikan output siap dinilai pada tahap assessing.

In [13]:
def parse_mixed_dates(series):
    values = series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

    try:
        return pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)
    except TypeError:
        parsed = pd.to_datetime(values, errors='coerce')
        missing_mask = parsed.isna() & values.notna()
        parsed.loc[missing_mask] = pd.to_datetime(values[missing_mask], errors='coerce', dayfirst=True)
        return parsed


def parse_amount(series):
    values = series.astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
    values = values.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    return pd.to_numeric(values, errors='coerce')


df_combined['date'] = parse_mixed_dates(df_combined['date'])
df_combined['amount'] = parse_amount(df_combined['amount'])
valid_dates = df_combined['date'].dropna()

print('Date range validation:')
print(f'Min date: {valid_dates.min()}')
print(f'Max date: {valid_dates.max()}')
print(f'Invalid dates: {df_combined["date"].isnull().sum():,}')
print('\nAmount validation:')
print(df_combined['amount'].describe())
print(f'Invalid amounts: {df_combined["amount"].isnull().sum():,}')
print(f'Negative amounts: {(df_combined["amount"] < 0).sum():,}')
print(f'Zero amounts: {(df_combined["amount"] == 0).sum():,}')

Date range validation:
Min date: 2015-01-01 00:00:00
Max date: 2025-11-30 23:17:00
Invalid dates: 10,808

Amount validation:
count    1.187130e+05
mean     2.679710e+04
std      1.075454e+05
min      0.000000e+00
25%      1.961600e+01
50%      1.441980e+02
75%      2.280000e+04
max      3.403591e+06
Name: amount, dtype: float64
Invalid amounts: 308
Negative amounts: 0
Zero amounts: 15,876


## Preview Final Dataset

Tujuan: melihat struktur akhir dataset sebelum export.

In [14]:
print('=== FINAL DATASET PREVIEW ===\n')
print(df_combined.info())
display(df_combined.head(10))

=== FINAL DATASET PREVIEW ===



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119021 entries, 0 to 119020
Data columns (total 66 columns):
 #   Column                                                      Non-Null Count   Dtype         
---  ------                                                      --------------   -----         
 0   date                                                        108213 non-null  datetime64[ns]
 1   amount                                                      118713 non-null  float64       
 2   category                                                    119021 non-null  object        
 3   description                                                 118500 non-null  object        
 4   data_source                                                 119021 non-null  object        
 5   order_id                                                    47106 non-null   object        
 6   total_qty                                                   62544 non-null   object        
 7   total_weigh

,date,amount,category,description,data_source,order_id,total_qty,total_weight_gr,total_returned_qty,total_diskon,product_categories,num_product_categories,status_pesanan,alasan_pembatalan,opsi_pengiriman,metode_pembayaran,kota_kabupaten,provinsi,ongkos_kirim_dibayar_oleh_pembeli,estimasi_potongan_biaya_pengiriman,total_pembayaran,perkiraan_ongkos_kirim,waktu_pesanan_dibuat,source_file,source_path,product_category,jumlah,returned_quantity,total_berat,waktu_pengiriman_diatur,status_pembatalan_pengembalian,no_resi,antar_ke_counter_pick_up,pesanan_harus_dikirimkan_sebelum_menghindari_keterlambatan,waktu_pembayaran_dilakukan,sku_induk,nomor_referensi_sku,nama_variasi,harga_awal,harga_setelah_diskon,total_harga_produk,diskon_dari_penjual,diskon_dari_shopee,berat_produk,jumlah_produk_di_pesan,voucher_ditanggung_penjual,cashback_koin,voucher_ditanggung_shopee,paket_diskon,paket_diskon_diskon_dari_shopee,paket_diskon_diskon_dari_penjual,potongan_koin_shopee,diskon_kartu_kredit,ongkos_kirim_pengembalian_barang,catatan_dari_pembeli,catatan,username_pembeli,nama_penerima,no_telepon,alamat_pengiriman,waktu_pesanan_selesai,mode,subcategory,income_expense,currency,type
0,2024-04-01 00:15:00,38300.0,Celengan,ORD_0000001,household_transaction,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-01 01:47:00,18576.0,Celengan,ORD_0000002,household_transaction,ORD_0000002,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-01 04:25:00,7069.0,Celengan,ORD_0000003,household_transaction,ORD_0000003,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-01 04:41:00,32200.0,Mangkok Sambal / Saus,ORD_0000004,household_transaction,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-01 06:12:00,0.0,"Keranjang, Other, Tempat Nasi",ORD_0000005,household_transaction,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-04-01 07:09:00,40996.0,Celengan,ORD_0000006,household_transaction,ORD_0000006,2,1000,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,SPayLater,KAB. BANDUNG,JAWA BARAT,0,10000,40996,10000,2024-04-01 07:09,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

## Save Combined Dataset

Tujuan: menyimpan dataset raw gabungan ke folder interim sebagai input tahap assessing.

In [15]:
output_path = interim_data_path / 'combined_raw_transactions.csv'
df_combined.to_csv(output_path, index=False)

print(f'Dataset saved to: {output_path}')
print(f'File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB')

Dataset saved to: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim/combined_raw_transactions.csv
File size: 40.98 MB
